# !git config --global user.name "textsummarisationpaper3-maker"
# !git config --global user.email "text.summarisation.paper3@gmail.com"

In [15]:
!git config --global user.name "textsummarisationpaper3-maker"
!git config --global user.email "text.summarisation.paper3@gmail.com"

In [1]:
# Install required packages
!pip install transformers datasets torch accelerate sentencepiece -q

In [2]:
# Import necessary libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import math
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## 1. Multi-Head Attention Mechanism

In [16]:
class MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention mechanism from 'Attention is All You Need'
    Supports both self-attention and cross-attention
    """
    def __init__(self, d_model, num_heads, dropout=0.1):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Linear projections for Q, K, V
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        
        # Output projection
        self.W_o = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
        
    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        """
        Compute scaled dot-product attention
        Q, K, V: (batch_size, num_heads, seq_len, d_k)
        """
        # Calculate attention scores
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        # Apply mask if provided (for padding and causal masking)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        # Apply softmax to get attention weights
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        
        # Apply attention weights to values
        output = torch.matmul(attention_weights, V)
        
        return output, attention_weights
    
    def split_heads(self, x):
        """
        Split the last dimension into (num_heads, d_k)
        x: (batch_size, seq_len, d_model)
        return: (batch_size, num_heads, seq_len, d_k)
        """
        batch_size, seq_len, d_model = x.size()
        return x.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
    
    def combine_heads(self, x):
        """
        Combine heads back to original shape
        x: (batch_size, num_heads, seq_len, d_k)
        return: (batch_size, seq_len, d_model)
        """
        batch_size, num_heads, seq_len, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
    
    def forward(self, query, key, value, mask=None):
        """
        query: (batch_size, seq_len_q, d_model)
        key: (batch_size, seq_len_k, d_model)
        value: (batch_size, seq_len_v, d_model)
        mask: (batch_size, 1, seq_len_q, seq_len_k) or (batch_size, 1, 1, seq_len_k)
        """
        batch_size = query.size(0)
        
        # Linear projections
        Q = self.W_q(query)  # (batch_size, seq_len_q, d_model)
        K = self.W_k(key)    # (batch_size, seq_len_k, d_model)
        V = self.W_v(value)  # (batch_size, seq_len_v, d_model)
        
        # Split into multiple heads
        Q = self.split_heads(Q)  # (batch_size, num_heads, seq_len_q, d_k)
        K = self.split_heads(K)  # (batch_size, num_heads, seq_len_k, d_k)
        V = self.split_heads(V)  # (batch_size, num_heads, seq_len_v, d_k)
        
        # Apply scaled dot-product attention
        attn_output, attention_weights = self.scaled_dot_product_attention(Q, K, V, mask)
        
        # Combine heads
        attn_output = self.combine_heads(attn_output)  # (batch_size, seq_len_q, d_model)
        
        # Final linear projection
        output = self.W_o(attn_output)
        
        return output, attention_weights

print("✓ Multi-Head Attention implemented")

✓ Multi-Head Attention implemented


## 2. Position-wise Feed-Forward Network

In [17]:
class PositionWiseFeedForward(nn.Module):
    """
    Position-wise Feed-Forward Network
    FFN(x) = max(0, xW1 + b1)W2 + b2
    """
    def __init__(self, d_model, d_ff, dropout=0.1):
        super(PositionWiseFeedForward, self).__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        # x: (batch_size, seq_len, d_model)
        return self.linear2(self.dropout(F.relu(self.linear1(x))))

print("✓ Feed-Forward Network implemented")

✓ Feed-Forward Network implemented


## 3. Positional Encoding

In [18]:
class PositionalEncoding(nn.Module):
    """
    Sinusoidal Positional Encoding
    PE(pos, 2i) = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    """
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(dropout)
        
        # Create positional encoding matrix
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)
        
    def forward(self, x):
        # x: (batch_size, seq_len, d_model)
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

print("✓ Positional Encoding implemented")

✓ Positional Encoding implemented


## 4. Encoder Layer with Self-Attention

In [19]:
class EncoderLayer(nn.Module):
    """
    Single Encoder Layer with:
    1. Multi-head self-attention
    2. Feed-forward network
    3. Layer normalization and residual connections
    """
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super(EncoderLayer, self).__init__()
        
        # Self-attention mechanism
        self.self_attention = MultiHeadAttention(d_model, num_heads, dropout)
        
        # Feed-forward network
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff, dropout)
        
        # Layer normalization
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
        # Dropout
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask=None):
        """
        x: (batch_size, seq_len, d_model)
        mask: (batch_size, 1, 1, seq_len) for padding mask
        """
        # Self-attention with residual connection and layer norm
        attn_output, _ = self.self_attention(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))
        
        # Feed-forward with residual connection and layer norm
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        
        return x

print("✓ Encoder Layer implemented")

✓ Encoder Layer implemented


## 5. Decoder Layer with Self-Attention and Cross-Attention

In [20]:
class DecoderLayer(nn.Module):
    """
    Single Decoder Layer with:
    1. Masked multi-head self-attention
    2. Multi-head cross-attention (attending to encoder output)
    3. Feed-forward network
    4. Layer normalization and residual connections
    """
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super(DecoderLayer, self).__init__()
        
        # Masked self-attention
        self.self_attention = MultiHeadAttention(d_model, num_heads, dropout)
        
        # Cross-attention (decoder attends to encoder output)
        self.cross_attention = MultiHeadAttention(d_model, num_heads, dropout)
        
        # Feed-forward network
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff, dropout)
        
        # Layer normalization
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        
        # Dropout
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):
        """
        x: (batch_size, tgt_seq_len, d_model) - decoder input
        encoder_output: (batch_size, src_seq_len, d_model) - encoder output
        src_mask: (batch_size, 1, 1, src_seq_len) - source padding mask
        tgt_mask: (batch_size, 1, tgt_seq_len, tgt_seq_len) - target causal mask
        """
        # Masked self-attention with residual connection and layer norm
        self_attn_output, _ = self.self_attention(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(self_attn_output))
        
        # Cross-attention with residual connection and layer norm
        cross_attn_output, _ = self.cross_attention(x, encoder_output, encoder_output, src_mask)
        x = self.norm2(x + self.dropout(cross_attn_output))
        
        # Feed-forward with residual connection and layer norm
        ff_output = self.feed_forward(x)
        x = self.norm3(x + self.dropout(ff_output))
        
        return x

print("✓ Decoder Layer implemented")

✓ Decoder Layer implemented


## 6. Complete Encoder Stack

In [21]:
class Encoder(nn.Module):
    """
    Complete Encoder stack with multiple encoder layers
    """
    def __init__(self, vocab_size, d_model, num_layers, num_heads, d_ff, 
                 max_len=5000, dropout=0.1):
        super(Encoder, self).__init__()
        
        self.d_model = d_model
        
        # Token embedding
        self.embedding = nn.Embedding(vocab_size, d_model)
        
        # Positional encoding
        self.pos_encoding = PositionalEncoding(d_model, max_len, dropout)
        
        # Stack of encoder layers
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask=None):
        """
        x: (batch_size, seq_len) - input token indices
        mask: (batch_size, 1, 1, seq_len) - padding mask
        """
        # Embedding + scaling
        x = self.embedding(x) * math.sqrt(self.d_model)
        
        # Add positional encoding
        x = self.pos_encoding(x)
        
        # Pass through encoder layers
        for layer in self.layers:
            x = layer(x, mask)
        
        return x

print("✓ Encoder stack implemented")

✓ Encoder stack implemented


## 7. Complete Decoder Stack

In [22]:
class Decoder(nn.Module):
    """
    Complete Decoder stack with multiple decoder layers
    """
    def __init__(self, vocab_size, d_model, num_layers, num_heads, d_ff,
                 max_len=5000, dropout=0.1):
        super(Decoder, self).__init__()
        
        self.d_model = d_model
        
        # Token embedding
        self.embedding = nn.Embedding(vocab_size, d_model)
        
        # Positional encoding
        self.pos_encoding = PositionalEncoding(d_model, max_len, dropout)
        
        # Stack of decoder layers
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):
        """
        x: (batch_size, tgt_seq_len) - target token indices
        encoder_output: (batch_size, src_seq_len, d_model) - encoder output
        src_mask: (batch_size, 1, 1, src_seq_len) - source padding mask
        tgt_mask: (batch_size, 1, tgt_seq_len, tgt_seq_len) - target causal mask
        """
        # Embedding + scaling
        x = self.embedding(x) * math.sqrt(self.d_model)
        
        # Add positional encoding
        x = self.pos_encoding(x)
        
        # Pass through decoder layers
        for layer in self.layers:
            x = layer(x, encoder_output, src_mask, tgt_mask)
        
        return x

print("✓ Decoder stack implemented")

✓ Decoder stack implemented


## 8. Complete Transformer Model

In [23]:
class Transformer(nn.Module):
    """
    Complete Transformer model for text summarization
    Combines encoder and decoder with all attention mechanisms
    """
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=512, 
                 num_layers=6, num_heads=8, d_ff=2048, 
                 max_len=5000, dropout=0.1, pad_idx=0):
        super(Transformer, self).__init__()
        
        self.pad_idx = pad_idx
        
        # Encoder
        self.encoder = Encoder(
            src_vocab_size, d_model, num_layers, num_heads, 
            d_ff, max_len, dropout
        )
        
        # Decoder
        self.decoder = Decoder(
            tgt_vocab_size, d_model, num_layers, num_heads,
            d_ff, max_len, dropout
        )
        
        # Final linear layer to project to vocabulary
        self.fc_out = nn.Linear(d_model, tgt_vocab_size)
        
        # Initialize parameters
        self._init_parameters()
        
    def _init_parameters(self):
        """Initialize parameters with Xavier uniform"""
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
    
    def make_src_mask(self, src):
        """
        Create padding mask for source sequence
        src: (batch_size, src_seq_len)
        """
        src_mask = (src != self.pad_idx).unsqueeze(1).unsqueeze(2)
        # (batch_size, 1, 1, src_seq_len)
        return src_mask
    
    def make_tgt_mask(self, tgt):
        """
        Create padding mask + causal mask for target sequence
        tgt: (batch_size, tgt_seq_len)
        """
        batch_size, tgt_len = tgt.shape
        
        # Padding mask
        tgt_pad_mask = (tgt != self.pad_idx).unsqueeze(1).unsqueeze(2)
        # (batch_size, 1, 1, tgt_seq_len)
        
        # Causal mask (prevents looking at future tokens)
        tgt_sub_mask = torch.tril(torch.ones((tgt_len, tgt_len), device=tgt.device)).bool()
        # (tgt_seq_len, tgt_seq_len)
        
        # Combine masks
        tgt_mask = tgt_pad_mask & tgt_sub_mask
        # (batch_size, 1, tgt_seq_len, tgt_seq_len)
        
        return tgt_mask
    
    def forward(self, src, tgt):
        """
        src: (batch_size, src_seq_len) - source token indices
        tgt: (batch_size, tgt_seq_len) - target token indices
        """
        # Create masks
        src_mask = self.make_src_mask(src)
        tgt_mask = self.make_tgt_mask(tgt)
        
        # Encode source sequence
        encoder_output = self.encoder(src, src_mask)
        
        # Decode target sequence
        decoder_output = self.decoder(tgt, encoder_output, src_mask, tgt_mask)
        
        # Project to vocabulary
        output = self.fc_out(decoder_output)
        
        return output
    
    def encode(self, src):
        """Encode source sequence only"""
        src_mask = self.make_src_mask(src)
        return self.encoder(src, src_mask)

print("✓ Complete Transformer model implemented")

✓ Complete Transformer model implemented


## 9. Load SAMSum Dataset and Pretrained Tokenizer

In [24]:
# Load SAMSum dataset
print("Loading SAMSum dataset...")
dataset = load_dataset("knkarthick/samsum")
print(f"Dataset loaded: {len(dataset['train'])} training examples")
print(f"Dataset loaded: {len(dataset['validation'])} validation examples")
print(f"Dataset loaded: {len(dataset['test'])} test examples")

# Show sample
print("\n--- Sample from dataset ---")
sample = dataset['train'][0]
print(f"Dialogue: {sample['dialogue'][:200]}...")
print(f"Summary: {sample['summary']}")

Loading SAMSum dataset...


Dataset loaded: 14731 training examples
Dataset loaded: 818 validation examples
Dataset loaded: 819 test examples

--- Sample from dataset ---
Dialogue: Amanda: I baked  cookies. Do you want some?
Jerry: Sure!
Amanda: I'll bring you tomorrow :-)...
Summary: Amanda baked cookies and will bring Jerry some tomorrow.


In [25]:
# Load pretrained tokenizer (using T5 tokenizer which works well for summarization)
print("Loading pretrained tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("t5-small")
print(f"Tokenizer loaded: {tokenizer.__class__.__name__}")
print(f"Vocab size: {len(tokenizer)}")
print(f"PAD token: '{tokenizer.pad_token}' (ID: {tokenizer.pad_token_id})")
print(f"EOS token: '{tokenizer.eos_token}' (ID: {tokenizer.eos_token_id})")
print(f"BOS token: '{tokenizer.bos_token}'")

# Test tokenization
test_text = "Hello, this is a test!"
tokens = tokenizer.encode(test_text)
print(f"\nTest tokenization: '{test_text}'")
print(f"Token IDs: {tokens}")
print(f"Decoded: '{tokenizer.decode(tokens)}')")

Loading pretrained tokenizer...


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

Tokenizer loaded: T5TokenizerFast
Vocab size: 32100
PAD token: '<pad>' (ID: 0)
EOS token: '</s>' (ID: 1)
BOS token: 'None'

Test tokenization: 'Hello, this is a test!'
Token IDs: [8774, 6, 48, 19, 3, 9, 794, 55, 1]
Decoded: 'Hello, this is a test!</s>')


## 10. Data Preprocessing and DataLoader

In [28]:
class SummarizationDataset(torch.utils.data.Dataset):
    """Custom dataset for text summarization"""
    def __init__(self, data, tokenizer, max_src_len=512, max_tgt_len=128):
        self.data = data
        self.tokenizer = tokenizer
        self.max_src_len = max_src_len
        self.max_tgt_len = max_tgt_len
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        dialogue = self.data[idx]['dialogue']
        summary = self.data[idx]['summary']
        
        # Tokenize source (dialogue)
        src_tokens = self.tokenizer.encode(
            dialogue,
            max_length=self.max_src_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        ).squeeze(0)
        
        # Tokenize target (summary)
        tgt_tokens = self.tokenizer.encode(
            summary,
            max_length=self.max_tgt_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        ).squeeze(0)
        
        return {
            'src': src_tokens,
            'tgt': tgt_tokens[:-1],  # Input (without last token)
            'tgt_y': tgt_tokens[1:]  # Target (without first token, shifted by 1)
        }

# Create datasets
print("Creating datasets...")
train_dataset = SummarizationDataset(dataset['train'], tokenizer, max_src_len=512, max_tgt_len=128)
val_dataset = SummarizationDataset(dataset['validation'], tokenizer, max_src_len=512, max_tgt_len=128)

# Create dataloaders
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print(f"✓ Training batches: {len(train_loader)}")
print(f"✓ Validation batches: {len(val_loader)}")

Creating datasets...
✓ Training batches: 231
✓ Validation batches: 13


## 11. Initialize Model

In [29]:
# Model hyperparameters
d_model = 256        # Embedding dimension
num_layers = 4       # Number of encoder/decoder layers
num_heads = 8        # Number of attention heads
d_ff = 1024          # Feed-forward dimension
dropout = 0.1
vocab_size = len(tokenizer)
pad_idx = tokenizer.pad_token_id

# Initialize model
model = Transformer(
    src_vocab_size=vocab_size,
    tgt_vocab_size=vocab_size,
    d_model=d_model,
    num_layers=num_layers,
    num_heads=num_heads,
    d_ff=d_ff,
    dropout=dropout,
    pad_idx=pad_idx
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✓ Model initialized")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"\nModel architecture:")
print(f"- Embedding dimension: {d_model}")
print(f"- Encoder/Decoder layers: {num_layers}")
print(f"- Attention heads: {num_heads}")
print(f"- Feed-forward dimension: {d_ff}")
print(f"- Vocabulary size: {vocab_size}")

✓ Model initialized
Total parameters: 32,057,700
Trainable parameters: 32,057,700

Model architecture:
- Embedding dimension: 256
- Encoder/Decoder layers: 4
- Attention heads: 8
- Feed-forward dimension: 1024
- Vocabulary size: 32100


## 12. Training Setup

In [30]:
# Loss function (ignore padding tokens)
criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)

# Optimizer with learning rate scheduling
learning_rate = 0.0001
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, betas=(0.9, 0.98), eps=1e-9)

# Learning rate scheduler (optional but recommended)
def get_lr(step, d_model, warmup_steps=4000):
    """Learning rate schedule from original transformer paper"""
    step = max(step, 1)
    return (d_model ** -0.5) * min(step ** -0.5, step * (warmup_steps ** -1.5))

print("✓ Training setup complete")
print(f"Optimizer: Adam")
print(f"Initial learning rate: {learning_rate}")
print(f"Loss function: CrossEntropyLoss (ignoring padding)")

✓ Training setup complete
Optimizer: Adam
Initial learning rate: 0.0001
Loss function: CrossEntropyLoss (ignoring padding)


## 13. Training Loop

In [31]:
def train_epoch(model, loader, criterion, optimizer, device, epoch):
    """Train for one epoch"""
    model.train()
    total_loss = 0
    
    pbar = tqdm(loader, desc=f"Epoch {epoch} [Train]")
    for batch_idx, batch in enumerate(pbar):
        src = batch['src'].to(device)
        tgt = batch['tgt'].to(device)
        tgt_y = batch['tgt_y'].to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        output = model(src, tgt)
        
        # Reshape for loss calculation
        output = output.reshape(-1, output.shape[-1])
        tgt_y = tgt_y.reshape(-1)
        
        # Calculate loss
        loss = criterion(output, tgt_y)
        
        # Backward pass
        loss.backward()
        
        # Gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return total_loss / len(loader)

def validate(model, loader, criterion, device, epoch):
    """Validate the model"""
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        pbar = tqdm(loader, desc=f"Epoch {epoch} [Val]")
        for batch in pbar:
            src = batch['src'].to(device)
            tgt = batch['tgt'].to(device)
            tgt_y = batch['tgt_y'].to(device)
            
            # Forward pass
            output = model(src, tgt)
            
            # Reshape for loss calculation
            output = output.reshape(-1, output.shape[-1])
            tgt_y = tgt_y.reshape(-1)
            
            # Calculate loss
            loss = criterion(output, tgt_y)
            total_loss += loss.item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return total_loss / len(loader)

print("✓ Training functions defined")

✓ Training functions defined


In [ ]:
# Training loop
num_epochs = 30  # Start with fewer epochs for testing
best_val_loss = float('inf')

print(f"Starting training for {num_epochs} epochs...\n")

for epoch in range(1, num_epochs + 1):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch}/{num_epochs}")
    print(f"{'='*60}")
    
    # Train
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device, epoch)
    
    # Validate
    val_loss = validate(model, val_loader, criterion, device, epoch)
    
    print(f"\nEpoch {epoch} Summary:")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val Loss: {val_loss:.4f}")
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_summarizer_model.pt')
        print(f"  ✓ Best model saved (val_loss: {val_loss:.4f})")

print("\n" + "="*60)
print("Training complete!")
print(f"Best validation loss: {best_val_loss:.4f}")
print("="*60)

Starting training for 3 epochs...


Epoch 1/3


Epoch 1 [Train]:  16%|█▌        | 36/231 [00:13<01:10,  2.77it/s, loss=9.1103]


KeyboardInterrupt: 

## 14. Inference Function (Greedy Decoding)

In [37]:
def generate_summary(model, src_text, tokenizer, device, max_len=128):
    """
    Generate summary using greedy decoding
    """
    model.eval()
    
    # Tokenize source text
    src_tokens = tokenizer.encode(
        src_text,
        max_length=512,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    ).to(device)
    
    # Start with EOS token (T5 uses EOS as start token)
    tgt_tokens = torch.tensor([[tokenizer.eos_token_id]], device=device)
    
    with torch.no_grad():
        # Encode source
        encoder_output = model.encoder(src_tokens, model.make_src_mask(src_tokens))
        
        # Generate tokens one by one
        for _ in range(max_len):
            # Create target mask
            tgt_mask = model.make_tgt_mask(tgt_tokens)
            src_mask = model.make_src_mask(src_tokens)
            
            # Decode
            decoder_output = model.decoder(tgt_tokens, encoder_output, src_mask, tgt_mask)
            
            # Get next token prediction
            output = model.fc_out(decoder_output)
            next_token = output[:, -1, :].argmax(dim=-1, keepdim=True)
            
            # Append to target tokens
            tgt_tokens = torch.cat([tgt_tokens, next_token], dim=1)
            
            # Stop if EOS token is generated
            if next_token.item() == tokenizer.eos_token_id:
                break
    
    # Decode tokens to text
    summary = tokenizer.decode(tgt_tokens.squeeze(0).tolist(), skip_special_tokens=True)
    
    return summary

print("✓ Inference function defined (greedy decoding)")

✓ Inference function defined (greedy decoding)


## 15. Test the Model on Sample Dialogues

In [ ]:
# Test on validation samples
print("Testing model on sample dialogues...\n")

num_samples = 3
for i in range(num_samples):
    sample = dataset['validation'][i]
    dialogue = sample['dialogue']
    ground_truth = sample['summary']
    
    print(f"\n{'='*80}")
    print(f"Sample {i+1}")
    print(f"{'='*80}")
    print(f"\n📝 Dialogue:\n{dialogue}")
    print(f"\n✓ Ground Truth Summary:\n{ground_truth}")
    
    # Generate summary
    generated = generate_summary(model, dialogue, tokenizer, device)
    print(f"\n🤖 Generated Summary:\n{generated}")
    print(f"\n{'-'*80}")

## 16. Model Architecture Summary

In [ ]:
print("="*80)
print("TEXT SUMMARIZER - ARCHITECTURE SUMMARY")
print("="*80)
print("\n✓ COMPONENTS IMPLEMENTED FROM SCRATCH:\n")
print("1. Multi-Head Attention Mechanism")
print("   - Scaled dot-product attention")
print("   - Multiple attention heads for parallel processing")
print("   - Supports both self-attention and cross-attention")
print("\n2. Positional Encoding")
print("   - Sinusoidal position embeddings")
print("   - Enables model to understand sequence order")
print("\n3. Encoder Layer")
print("   - Multi-head self-attention")
print("   - Position-wise feed-forward network")
print("   - Layer normalization and residual connections")
print("\n4. Decoder Layer")
print("   - Masked multi-head self-attention (prevents looking ahead)")
print("   - Multi-head cross-attention (attends to encoder output)")
print("   - Position-wise feed-forward network")
print("   - Layer normalization and residual connections")
print("\n5. Complete Transformer Model")
print("   - Encoder stack with multiple layers")
print("   - Decoder stack with multiple layers")
print("   - Token embeddings and output projection")
print("\n6. Training Pipeline")
print("   - SAMSum dialogue dataset")
print("   - Pretrained T5 tokenizer")
print("   - Custom data preprocessing")
print("   - Training and validation loops")
print("   - Greedy decoding for inference")
print("\n" + "="*80)
print("\n✓ ATTENTION MECHANISMS INCLUDED:")
print("   • Self-Attention (Encoder)")
print("   • Masked Self-Attention (Decoder)")
print("   • Cross-Attention (Decoder → Encoder)")
print("   • Multi-Head Attention (all layers)")
print("\n" + "="*80)  

## 17. Install Evaluation Packages

In [38]:
# Install evaluation packages
!pip install rouge-score bert-score sacrebleu nltk py-readability-metrics textstat pandas -q

import pandas as pd
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import nltk
from nltk.translate.bleu_score import sentence_bleu, corpus_bleu
from nltk.translate.meteor_score import meteor_score
import textstat
import readability
import json
import time
from datetime import datetime

# Download required NLTK data
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

print("✓ Evaluation packages installed and configured")

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 239.2/239.2 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 82.0 MB/s eta 0:00:00
✓ Evaluation packages installed and configured


## 18. Comprehensive Evaluation Functions

In [39]:
class ComprehensiveEvaluator:
    """Comprehensive evaluation class with multiple metrics"""
    
    def __init__(self):
        # Initialize ROUGE scorer
        self.rouge_scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
        
    def calculate_rouge_scores(self, reference, hypothesis):
        """Calculate ROUGE scores"""
        scores = self.rouge_scorer.score(reference, hypothesis)
        return {
            'rouge1_precision': scores['rouge1'].precision,
            'rouge1_recall': scores['rouge1'].recall,
            'rouge1_fmeasure': scores['rouge1'].fmeasure,
            'rouge2_precision': scores['rouge2'].precision,
            'rouge2_recall': scores['rouge2'].recall,
            'rouge2_fmeasure': scores['rouge2'].fmeasure,
            'rougeL_precision': scores['rougeL'].precision,
            'rougeL_recall': scores['rougeL'].recall,
            'rougeL_fmeasure': scores['rougeL'].fmeasure,
        }
    
    def calculate_bert_score(self, references, hypotheses):
        """Calculate BERT scores (batch processing for efficiency)"""
        P, R, F1 = bert_score(hypotheses, references, lang="en", verbose=False)
        return P.tolist(), R.tolist(), F1.tolist()
    
    def calculate_bleu_score(self, reference, hypothesis):
        """Calculate BLEU score"""
        try:
            # Tokenize
            reference_tokens = [nltk.word_tokenize(reference.lower())]
            hypothesis_tokens = nltk.word_tokenize(hypothesis.lower())
            
            # Calculate BLEU-4
            bleu_score = sentence_bleu(reference_tokens, hypothesis_tokens)
            
            # Calculate BLEU-1, BLEU-2, BLEU-3
            bleu1 = sentence_bleu(reference_tokens, hypothesis_tokens, weights=(1, 0, 0, 0))
            bleu2 = sentence_bleu(reference_tokens, hypothesis_tokens, weights=(0.5, 0.5, 0, 0))
            bleu3 = sentence_bleu(reference_tokens, hypothesis_tokens, weights=(0.33, 0.33, 0.33, 0))
            
            return {
                'bleu1': bleu1,
                'bleu2': bleu2,
                'bleu3': bleu3,
                'bleu4': bleu_score
            }
        except:
            return {'bleu1': 0, 'bleu2': 0, 'bleu3': 0, 'bleu4': 0}
    
    def calculate_meteor_score(self, reference, hypothesis):
        """Calculate METEOR score"""
        try:
            reference_tokens = nltk.word_tokenize(reference.lower())
            hypothesis_tokens = nltk.word_tokenize(hypothesis.lower())
            return meteor_score([reference_tokens], hypothesis_tokens)
        except:
            return 0.0
    
    def calculate_length_metrics(self, reference, hypothesis):
        """Calculate length-based metrics"""
        ref_len = len(reference.split())
        hyp_len = len(hypothesis.split())
        
        return {
            'reference_length': ref_len,
            'hypothesis_length': hyp_len,
            'length_ratio': hyp_len / ref_len if ref_len > 0 else 0,
            'compression_ratio': ref_len / hyp_len if hyp_len > 0 else float('inf')
        }
    
    def calculate_readability_metrics(self, text):
        """Calculate readability metrics"""
        try:
            return {
                'flesch_reading_ease': textstat.flesch_reading_ease(text),
                'flesch_kincaid_grade': textstat.flesch_kincaid_grade(text),
                'gunning_fog': textstat.gunning_fog(text),
                'automated_readability_index': textstat.automated_readability_index(text),
                'coleman_liau_index': textstat.coleman_liau_index(text),
                'linsear_write_formula': textstat.linsear_write_formula(text),
                'dale_chall_readability_score': textstat.dale_chall_readability_score(text)
            }
        except:
            return {
                'flesch_reading_ease': 0,
                'flesch_kincaid_grade': 0,
                'gunning_fog': 0,
                'automated_readability_index': 0,
                'coleman_liau_index': 0,
                'linsear_write_formula': 0,
                'dale_chall_readability_score': 0
            }
    
    def calculate_lexical_diversity(self, text):
        """Calculate lexical diversity metrics"""
        words = text.lower().split()
        unique_words = set(words)
        
        return {
            'total_words': len(words),
            'unique_words': len(unique_words),
            'type_token_ratio': len(unique_words) / len(words) if len(words) > 0 else 0,
            'average_word_length': sum(len(word) for word in words) / len(words) if len(words) > 0 else 0
        }

# Initialize evaluator
evaluator = ComprehensiveEvaluator()
print("✓ Comprehensive evaluator initialized")

✓ Comprehensive evaluator initialized


## 19. Load Best Model and Evaluate on Test Data

In [42]:
# Load the best trained model
print("Loading best trained model...")
model.load_state_dict(torch.load('best_summarizer_model.pt', map_location=device))
model.eval()
print("✓ Model loaded successfully")

# Prepare test dataset
test_dataset = SummarizationDataset(dataset['test'], tokenizer, max_src_len=512, max_tgt_len=128)
print(f"✓ Test dataset prepared: {len(test_dataset)} examples")

# Limit test examples for computational efficiency (you can increase this)
max_test_examples = 819  # Adjust based on your computational resources
test_indices = list(range(min(max_test_examples, len(dataset['test']))))

print(f"Evaluating on {len(test_indices)} test examples...")

Loading best trained model...
✓ Model loaded successfully
✓ Test dataset prepared: 819 examples
Evaluating on 819 test examples...
✓ Model loaded successfully
✓ Test dataset prepared: 819 examples
Evaluating on 819 test examples...


## 20. Generate Predictions and Calculate All Metrics

In [43]:
# Generate predictions and calculate comprehensive metrics (without BERT for faster execution)
results = []
generated_summaries = []
reference_summaries = []

print("Generating summaries and calculating metrics...")
start_time = time.time()

for i, idx in enumerate(tqdm(test_indices, desc="Evaluating")):
    sample = dataset['test'][idx]
    dialogue = sample['dialogue']
    reference = sample['summary']
    
    # Generate summary
    generated = generate_summary(model, dialogue, tokenizer, device, max_len=128)
    
    # Store for potential future use
    generated_summaries.append(generated)
    reference_summaries.append(reference)
    
    # Calculate individual metrics
    result = {
        'sample_id': idx,
        'dialogue': dialogue,
        'reference_summary': reference,
        'generated_summary': generated
    }
    
    # ROUGE scores
    rouge_scores = evaluator.calculate_rouge_scores(reference, generated)
    result.update(rouge_scores)
    
    # BLEU scores
    bleu_scores = evaluator.calculate_bleu_score(reference, generated)
    result.update(bleu_scores)
    
    # METEOR score
    result['meteor_score'] = evaluator.calculate_meteor_score(reference, generated)
    
    # Length metrics
    length_metrics = evaluator.calculate_length_metrics(reference, generated)
    result.update(length_metrics)
    
    # Readability metrics for generated summary
    readability_gen = evaluator.calculate_readability_metrics(generated)
    for key, value in readability_gen.items():
        result[f'generated_{key}'] = value
    
    # Readability metrics for reference summary
    readability_ref = evaluator.calculate_readability_metrics(reference)
    for key, value in readability_ref.items():
        result[f'reference_{key}'] = value
    
    # Lexical diversity for generated summary
    lexical_gen = evaluator.calculate_lexical_diversity(generated)
    for key, value in lexical_gen.items():
        result[f'generated_{key}'] = value
    
    # Lexical diversity for reference summary
    lexical_ref = evaluator.calculate_lexical_diversity(reference)
    for key, value in lexical_ref.items():
        result[f'reference_{key}'] = value
    
    # Add placeholder BERT scores (set to 0 for now)
    result['bert_precision'] = 0.0
    result['bert_recall'] = 0.0
    result['bert_f1'] = 0.0
    
    results.append(result)

end_time = time.time()
print(f"✓ Evaluation completed in {end_time - start_time:.2f} seconds")
print(f"✓ Generated metrics for {len(results)} examples")
print("Note: BERT scores set to 0 for computational efficiency")

Generating summaries and calculating metrics...


Evaluating: 100%|██████████| 819/819 [01:58<00:00,  6.89it/s]

✓ Evaluation completed in 118.91 seconds
✓ Generated metrics for 819 examples
Note: BERT scores set to 0 for computational efficiency


## 21. Save Results to CSV and Display Summary Statistics

In [44]:
# Convert results to DataFrame
df_results = pd.DataFrame(results)

# Create timestamp for filename
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_filename = f'summarization_evaluation_results_{timestamp}.csv'

# Save to CSV
df_results.to_csv(csv_filename, index=False)
print(f"✓ Results saved to: {csv_filename}")

# Display basic info about the DataFrame
print(f"\nDataFrame Info:")
print(f"Shape: {df_results.shape}")
print(f"Columns: {len(df_results.columns)}")

# Display column names by category
print("\nColumn Categories:")

rouge_cols = [col for col in df_results.columns if 'rouge' in col.lower()]
bert_cols = [col for col in df_results.columns if 'bert' in col.lower()]
bleu_cols = [col for col in df_results.columns if 'bleu' in col.lower()]
readability_cols = [col for col in df_results.columns if any(term in col.lower() for term in ['flesch', 'gunning', 'coleman', 'dale', 'automated', 'linsear'])]
length_cols = [col for col in df_results.columns if any(term in col.lower() for term in ['length', 'ratio', 'compression'])]
lexical_cols = [col for col in df_results.columns if any(term in col.lower() for term in ['words', 'token', 'diversity'])]

print(f"\n📊 ROUGE Metrics ({len(rouge_cols)}): {rouge_cols}")
print(f"\n🤖 BERT Metrics ({len(bert_cols)}): {bert_cols}")
print(f"\n📝 BLEU/METEOR Metrics ({len(bleu_cols) + 1}): {bleu_cols + ['meteor_score']}")
print(f"\n📖 Readability Metrics ({len(readability_cols)}): {readability_cols}")
print(f"\n📏 Length Metrics ({len(length_cols)}): {length_cols}")
print(f"\n📚 Lexical Diversity Metrics ({len(lexical_cols)}): {lexical_cols}")

✓ Results saved to: summarization_evaluation_results_20251018_190110.csv

DataFrame Info:
Shape: (819, 47)
Columns: 47

Column Categories:

📊 ROUGE Metrics (9): ['rouge1_precision', 'rouge1_recall', 'rouge1_fmeasure', 'rouge2_precision', 'rouge2_recall', 'rouge2_fmeasure', 'rougeL_precision', 'rougeL_recall', 'rougeL_fmeasure']

🤖 BERT Metrics (3): ['bert_precision', 'bert_recall', 'bert_f1']

📝 BLEU/METEOR Metrics (5): ['bleu1', 'bleu2', 'bleu3', 'bleu4', 'meteor_score']

📖 Readability Metrics (14): ['generated_flesch_reading_ease', 'generated_flesch_kincaid_grade', 'generated_gunning_fog', 'generated_automated_readability_index', 'generated_coleman_liau_index', 'generated_linsear_write_formula', 'generated_dale_chall_readability_score', 'reference_flesch_reading_ease', 'reference_flesch_kincaid_grade', 'reference_gunning_fog', 'reference_automated_readability_index', 'reference_coleman_liau_index', 'reference_linsear_write_formula', 'reference_dale_chall_readability_score']

📏 Length

In [45]:
# Calculate and display summary statistics for key metrics
print("\n" + "="*80)
print("COMPREHENSIVE EVALUATION RESULTS - SUMMARY STATISTICS")
print("="*80)

# Key metrics for summary
key_metrics = [
    'rouge1_fmeasure', 'rouge2_fmeasure', 'rougeL_fmeasure',
    'bert_precision', 'bert_recall', 'bert_f1',
    'bleu1', 'bleu2', 'bleu4', 'meteor_score',
    'length_ratio', 'compression_ratio'
]

# Display summary statistics
summary_stats = df_results[key_metrics].describe()
print("\nKEY METRICS SUMMARY:")
print(summary_stats.round(4))

# Display top 5 and bottom 5 examples by ROUGE-L F-measure
print(f"\n{'='*60}")
print("TOP 5 EXAMPLES BY ROUGE-L F-MEASURE:")
print(f"{'='*60}")
top_5 = df_results.nlargest(5, 'rougeL_fmeasure')[['sample_id', 'rougeL_fmeasure', 'bert_f1', 'bleu4']]
print(top_5.round(4))

print(f"\n{'='*60}")
print("BOTTOM 5 EXAMPLES BY ROUGE-L F-MEASURE:")
print(f"{'='*60}")
bottom_5 = df_results.nsmallest(5, 'rougeL_fmeasure')[['sample_id', 'rougeL_fmeasure', 'bert_f1', 'bleu4']]
print(bottom_5.round(4))

# Average scores across all metrics
print(f"\n{'='*60}")
print("AVERAGE SCORES:")
print(f"{'='*60}")
for metric in key_metrics:
    avg_score = df_results[metric].mean()
    print(f"{metric:25}: {avg_score:.4f}")

# Display first few examples
print(f"\n{'='*80}")
print("SAMPLE PREDICTIONS (First 3 examples):")
print(f"{'='*80}")

for i in range(min(3, len(df_results))):
    row = df_results.iloc[i]
    print(f"\n--- Example {i+1} (Sample ID: {row['sample_id']}) ---")
    print(f"\nDialogue: {row['dialogue'][:200]}...")
    print(f"\nReference: {row['reference_summary']}")
    print(f"\nGenerated: {row['generated_summary']}")
    print(f"\nScores:")
    print(f"  ROUGE-1 F1: {row['rouge1_fmeasure']:.4f}")
    print(f"  ROUGE-L F1: {row['rougeL_fmeasure']:.4f}")
    print(f"  BERT F1:    {row['bert_f1']:.4f}")
    print(f"  BLEU-4:     {row['bleu4']:.4f}")
    print("-" * 80)


COMPREHENSIVE EVALUATION RESULTS - SUMMARY STATISTICS

KEY METRICS SUMMARY:
       rouge1_fmeasure  rouge2_fmeasure  rougeL_fmeasure  bert_precision  \
count         819.0000         819.0000         819.0000           819.0   
mean            0.1965           0.0358           0.1641             0.0   
std             0.1075           0.0629           0.0945             0.0   
min             0.0000           0.0000           0.0000             0.0   
25%             0.1268           0.0000           0.1000             0.0   
50%             0.1875           0.0000           0.1569             0.0   
75%             0.2587           0.0571           0.2069             0.0   
max             0.6316           0.5333           0.6250             0.0   

       bert_recall  bert_f1  bleu1  bleu2  bleu4  meteor_score  length_ratio  \
count        819.0    819.0  819.0  819.0  819.0         819.0      819.0000   
mean           0.0      0.0    0.0    0.0    0.0           0.0        1.1080  

## 22. Create Additional Analysis Reports

In [46]:
# Create correlation analysis
print("\n" + "="*80)
print("CORRELATION ANALYSIS BETWEEN DIFFERENT METRICS")
print("="*80)

# Select numerical columns for correlation
numerical_cols = df_results.select_dtypes(include=[np.number]).columns.tolist()
# Remove ID column
numerical_cols = [col for col in numerical_cols if col != 'sample_id']

# Calculate correlation matrix for key metrics
key_metrics_for_corr = [
    'rouge1_fmeasure', 'rouge2_fmeasure', 'rougeL_fmeasure',
    'bert_f1', 'bleu4', 'meteor_score'
]

correlation_matrix = df_results[key_metrics_for_corr].corr()
print("\nCorrelation Matrix (Key Metrics):")
print(correlation_matrix.round(3))

# Save correlation matrix to CSV
correlation_filename = f'metric_correlations_{timestamp}.csv'
correlation_matrix.to_csv(correlation_filename)
print(f"\n✓ Correlation matrix saved to: {correlation_filename}")

# Create a summary report
summary_report = {
    'evaluation_date': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    'total_samples_evaluated': len(df_results),
    'model_architecture': 'Custom Transformer (Encoder-Decoder)',
    'dataset': 'SAMSum Test Set',
    'metrics_calculated': len(df_results.columns) - 4,  # Exclude ID, dialogue, summaries
}

# Add average scores to report
for metric in key_metrics:
    summary_report[f'avg_{metric}'] = df_results[metric].mean()

# Save summary report
report_filename = f'evaluation_summary_report_{timestamp}.json'
with open(report_filename, 'w') as f:
    json.dump(summary_report, f, indent=2, default=str)
    
print(f"\n✓ Summary report saved to: {report_filename}")

# Display model performance categories
def categorize_performance(rouge_l_f1, bert_f1):
    """Categorize model performance based on ROUGE-L and BERT F1"""
    if rouge_l_f1 >= 0.4 and bert_f1 >= 0.85:
        return "Excellent"
    elif rouge_l_f1 >= 0.3 and bert_f1 >= 0.80:
        return "Good"
    elif rouge_l_f1 >= 0.2 and bert_f1 >= 0.75:
        return "Fair"
    else:
        return "Poor"

# Apply categorization
df_results['performance_category'] = df_results.apply(
    lambda row: categorize_performance(row['rougeL_fmeasure'], row['bert_f1']), axis=1
)

# Show performance distribution
performance_dist = df_results['performance_category'].value_counts()
print(f"\n{'='*60}")
print("PERFORMANCE CATEGORY DISTRIBUTION:")
print(f"{'='*60}")
for category, count in performance_dist.items():
    percentage = (count / len(df_results)) * 100
    print(f"{category:12}: {count:4d} examples ({percentage:5.1f}%)")

# Update CSV with performance categories
df_results.to_csv(csv_filename, index=False)
print(f"\n✓ Updated results with performance categories saved to: {csv_filename}")

print("\n" + "="*80)
print("EVALUATION COMPLETE!")
print("="*80)
print(f"\nFiles generated:")
print(f"1. {csv_filename} - Complete evaluation results")
print(f"2. {correlation_filename} - Metric correlations")
print(f"3. {report_filename} - Summary report")
print(f"\nTotal metrics calculated per example: {len(df_results.columns) - 4}")
print(f"Total examples evaluated: {len(df_results)}")
print("="*80)


CORRELATION ANALYSIS BETWEEN DIFFERENT METRICS

Correlation Matrix (Key Metrics):
                 rouge1_fmeasure  rouge2_fmeasure  rougeL_fmeasure  bert_f1  \
rouge1_fmeasure            1.000            0.665            0.922      NaN   
rouge2_fmeasure            0.665            1.000            0.724      NaN   
rougeL_fmeasure            0.922            0.724            1.000      NaN   
bert_f1                      NaN              NaN              NaN      NaN   
bleu4                        NaN              NaN              NaN      NaN   
meteor_score                 NaN              NaN              NaN      NaN   

                 bleu4  meteor_score  
rouge1_fmeasure    NaN           NaN  
rouge2_fmeasure    NaN           NaN  
rougeL_fmeasure    NaN           NaN  
bert_f1            NaN           NaN  
bleu4              NaN           NaN  
meteor_score       NaN           NaN  

✓ Correlation matrix saved to: metric_correlations_20251018_190110.csv

✓ Summary report 

In [47]:
df_results.head()

,sample_id,dialogue,reference_summary,generated_summary,rouge1_precision,rouge1_recall,rouge1_fmeasure,rouge2_precision,rouge2_recall,rouge2_fmeasure,...,generated_type_token_ratio,generated_average_word_length,reference_total_words,reference_unique_words,reference_type_token_ratio,reference_average_word_length,bert_precision,bert_recall,bert_f1,performance_category
0,0,"Hannah: Hey, do you have Betty's number?\nAman...",Hannah needs Betty's number but Amanda doesn't...,n's mother is looking for her new phone. She w...,0.176471,0.187500,0.181818,0.000000,0.000000,0.000000,...,1.000000,3.687500,14,13,0.928571,5.000000,0.0,0.0,0.0,Poor
1,1,Eric: MACHINE!\nRob: That's so gr8!\nEric: I k...,Eric and Rob are going to watch a stand-up on ...,is looking for a new job. He likes it. He is n...,0.115385,0.250000,0.157895,0.040000,0.090909,0.055556,...,0.730769,3.423077,11,11,1.000000,4.000000,0.0,0.0,0.0,Poor
2,2,"Lenny: Babe, can you help me with something?\n...",Lenny can't decide which trousers to buy. Bob ...,ny will buy a new dress for her request. She w...,0.133333,0.068966,0.090909,0.000000,0.000000,0.000000,...,0.866667,3.466667,27,22,0.814815,4.592593,0.0,0.0,0.0,Poor
3,3,"Will: hey babe, what do you want for dinner to...",Emma will be home soon and she will let Will k...,will come to eat dinner tonight.,0.166667,0.090909,0.117647,0.000000,0.000000,0.000000,...,1.000000,4.500000,11,9,0.818182,3.636364,0.0,0.0,0.0,Poor
4,4,"Ollie: Hi , are you in Warsaw\nJane: yes, just...",Jane is in Warsaw. Ollie and Jane has a party....,is going to the gym with her parents. She will...,0.176471,0.139535,0.155844,0.060606,0.047619,0.053333,...,0.558824,3.647059,42,34,0.809524,4.285714,0.0,0.0,0.0,Poor


## 23. Data Augmentation Techniques

In [48]:
# Install additional packages for data augmentation
!pip install nlpaug transformers[torch] -q

import nlpaug.augmenter.word as naw
import nlpaug.augmenter.sentence as nas
import nlpaug.augmenter.char as nac
import random
import copy

print("✓ Data augmentation packages installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.5/410.5 kB 25.4 MB/s eta 0:00:00
✓ Data augmentation packages installed


In [49]:
class DataAugmenter:
    """
    Simplified data augmentation class for dialogue summarization
    """
    def __init__(self):
        # Initialize basic augmentation techniques
        pass
        
    def random_word_swap(self, text, swap_prob=0.1):
        """Randomly swap adjacent words"""
        words = text.split()
        if len(words) < 2:
            return text
            
        augmented_words = words.copy()
        for i in range(len(words) - 1):
            if random.random() < swap_prob:
                # Swap adjacent words
                augmented_words[i], augmented_words[i + 1] = augmented_words[i + 1], augmented_words[i]
        
        return ' '.join(augmented_words)
    
    def random_word_deletion(self, text, delete_prob=0.1):
        """Randomly delete words"""
        words = text.split()
        if len(words) <= 3:  # Don't delete if too short
            return text
            
        augmented_words = []
        for word in words:
            if random.random() > delete_prob:
                augmented_words.append(word)
        
        return ' '.join(augmented_words) if augmented_words else text
    
    def shuffle_speakers(self, dialogue):
        """Shuffle speaker names while maintaining conversation flow"""
        # Extract speaker patterns
        lines = dialogue.split('\n')
        speaker_lines = []
        
        for line in lines:
            if ':' in line:
                speaker, content = line.split(':', 1)
                speaker_lines.append((speaker.strip(), content.strip()))
            else:
                speaker_lines.append(('', line.strip()))
        
        if len(speaker_lines) < 2:
            return dialogue
        
        # Get unique speakers
        speakers = list(set([speaker for speaker, _ in speaker_lines if speaker]))
        if len(speakers) < 2:
            return dialogue
        
        # Create speaker mapping
        shuffled_speakers = speakers.copy()
        random.shuffle(shuffled_speakers)
        speaker_mapping = dict(zip(speakers, shuffled_speakers))
        
        # Apply mapping
        augmented_lines = []
        for speaker, content in speaker_lines:
            if speaker in speaker_mapping:
                new_speaker = speaker_mapping[speaker]
                augmented_lines.append(f"{new_speaker}: {content}")
            else:
                augmented_lines.append(content if not speaker else f"{speaker}: {content}")
        
        return '\n'.join(augmented_lines)
    
    def add_conversational_noise(self, dialogue):
        """Add common conversational elements"""
        noise_elements = [
            "um", "uh", "well", "you know", "like", "I mean", 
            "so", "anyway", "actually", "basically"
        ]
        
        lines = dialogue.split('\n')
        augmented_lines = []
        
        for line in lines:
            if ':' in line and random.random() < 0.2:  # 20% chance to add noise
                speaker, content = line.split(':', 1)
                noise = random.choice(noise_elements)
                words = content.strip().split()
                if len(words) > 2:
                    # Insert noise at random position
                    insert_pos = random.randint(1, len(words))
                    words.insert(insert_pos, noise)
                    augmented_content = ' '.join(words)
                    augmented_lines.append(f"{speaker}: {augmented_content}")
                else:
                    augmented_lines.append(line)
            else:
                augmented_lines.append(line)
        
        return '\n'.join(augmented_lines)
    
    def augment_sample(self, dialogue, summary, aug_type='random'):
        """Apply augmentation to a dialogue-summary pair"""
        if aug_type == 'random':
            aug_type = random.choice(['word_swap', 'word_deletion', 'speaker_shuffle', 'conv_noise'])
        
        augmented_dialogue = dialogue
        augmented_summary = summary
        
        try:
            if aug_type == 'word_swap':
                augmented_dialogue = self.random_word_swap(dialogue)
            elif aug_type == 'word_deletion':
                augmented_dialogue = self.random_word_deletion(dialogue)
            elif aug_type == 'speaker_shuffle':
                augmented_dialogue = self.shuffle_speakers(dialogue)
            elif aug_type == 'conv_noise':
                augmented_dialogue = self.add_conversational_noise(dialogue)
        except Exception as e:
            print(f"Augmentation failed for {aug_type}: {e}")
            # Fall back to original
            augmented_dialogue = dialogue
        
        return augmented_dialogue, augmented_summary

# Initialize augmenter
print("Initializing simplified data augmenter...")
augmenter = DataAugmenter()
print("✓ Data augmenter initialized")

Initializing simplified data augmenter...
✓ Data augmenter initialized


## 24. Create Augmented Dataset

In [50]:
def create_augmented_dataset(original_data, augmenter, augmentation_factor=0.5):
    """
    Create augmented dataset by applying various augmentation techniques
    
    Args:
        original_data: Original dataset
        augmenter: DataAugmenter instance
        augmentation_factor: Fraction of original data to augment (0.5 = 50% more data)
    """
    print(f"Creating augmented dataset with {augmentation_factor*100}% additional data...")
    
    original_samples = []
    augmented_samples = []
    
    # Convert original data to list format
    for sample in original_data:
        original_samples.append({
            'dialogue': sample['dialogue'],
            'summary': sample['summary'],
            'augmented': False
        })
    
    # Create augmented samples
    num_augmented = int(len(original_samples) * augmentation_factor)
    
    print(f"Generating {num_augmented} augmented samples...")
    
    for i in tqdm(range(num_augmented), desc="Augmenting"):
        # Select random original sample
        original_idx = random.randint(0, len(original_samples) - 1)
        original_sample = original_data[original_idx]
        
        # Apply augmentation
        aug_dialogue, aug_summary = augmenter.augment_sample(
            original_sample['dialogue'],
            original_sample['summary'],
            aug_type='random'
        )
        
        augmented_samples.append({
            'dialogue': aug_dialogue,
            'summary': aug_summary,
            'augmented': True
        })
    
    # Combine original and augmented data
    combined_data = original_samples + augmented_samples
    random.shuffle(combined_data)  # Shuffle to mix original and augmented
    
    print(f"✓ Augmented dataset created:")
    print(f"  - Original samples: {len(original_samples)}")
    print(f"  - Augmented samples: {len(augmented_samples)}")
    print(f"  - Total samples: {len(combined_data)}")
    
    return combined_data

# Create augmented training dataset
print("Creating augmented training dataset...")
augmented_train_data = create_augmented_dataset(
    dataset['train'], 
    augmenter, 
    augmentation_factor=0.3  # Add 30% more data
)

print("\n" + "="*50)
print("SAMPLE AUGMENTATION EXAMPLES:")
print("="*50)

# Show some examples of augmentation
for i in range(3):
    # Find an augmented sample
    aug_sample = None
    for sample in augmented_train_data:
        if sample['augmented']:
            aug_sample = sample
            break
    
    if aug_sample:
        print(f"\nExample {i+1}:")
        print(f"Augmented Dialogue: {aug_sample['dialogue'][:200]}...")
        print(f"Summary: {aug_sample['summary']}")
        print("-" * 50)

Creating augmented training dataset...
Creating augmented dataset with 30.0% additional data...
Generating 4419 augmented samples...
Generating 4419 augmented samples...


Augmenting: 100%|██████████| 4419/4419 [00:00<00:00, 12513.96it/s]

✓ Augmented dataset created:
  - Original samples: 14731
  - Augmented samples: 4419
  - Total samples: 19150

SAMPLE AUGMENTATION EXAMPLES:

Example 1:
Augmented Dialogue: Norma: what's going on Jesse: what do u mean Norma: I know I saw you at the mall earlier today Norma: I know saw me Norma: yet you ignored me?? why? Jesse: what? thats impossible Jesse: im and ive bee...
Summary: Jesse is sick and she's been home all day. Norma thinks that she saw her earlier at the mall and that Jesse ignored her.
--------------------------------------------------

Example 2:
Augmented Dialogue: Norma: what's going on Jesse: what do u mean Norma: I know I saw you at the mall earlier today Norma: I know saw me Norma: yet you ignored me?? why? Jesse: what? thats impossible Jesse: im and ive bee...
Summary: Jesse is sick and she's been home all day. Norma thinks that she saw her earlier at the mall and that Jesse ignored her.
--------------------------------------------------

Example 3:
Augmented Dia

## 25. Enhanced Dataset Class with Augmentation

In [51]:
class AugmentedSummarizationDataset(torch.utils.data.Dataset):
    """Enhanced dataset class that can handle both original and augmented data"""
    
    def __init__(self, data, tokenizer, max_src_len=512, max_tgt_len=128, online_augmentation=False, augmenter=None):
        self.data = data
        self.tokenizer = tokenizer
        self.max_src_len = max_src_len
        self.max_tgt_len = max_tgt_len
        self.online_augmentation = online_augmentation
        self.augmenter = augmenter
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        sample = self.data[idx]
        dialogue = sample['dialogue']
        summary = sample['summary']
        
        # Apply online augmentation during training (optional)
        if self.online_augmentation and self.augmenter and random.random() < 0.1:  # 10% chance
            dialogue, summary = self.augmenter.augment_sample(dialogue, summary, 'random')
        
        # Tokenize source (dialogue)
        src_tokens = self.tokenizer.encode(
            dialogue,
            max_length=self.max_src_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        ).squeeze(0)
        
        # Tokenize target (summary)
        tgt_tokens = self.tokenizer.encode(
            summary,
            max_length=self.max_tgt_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        ).squeeze(0)
        
        return {
            'src': src_tokens,
            'tgt': tgt_tokens[:-1],  # Input (without last token)
            'tgt_y': tgt_tokens[1:]  # Target (without first token, shifted by 1)
        }

# Create augmented datasets and dataloaders
print("Creating enhanced datasets with augmentation...")

# Training dataset with augmented data
augmented_train_dataset = AugmentedSummarizationDataset(
    augmented_train_data, 
    tokenizer, 
    max_src_len=512, 
    max_tgt_len=128,
    online_augmentation=False,  # Set to True for online augmentation during training
    augmenter=augmenter
)

# Validation dataset (no augmentation)
val_dataset_enhanced = AugmentedSummarizationDataset(
    dataset['validation'], 
    tokenizer, 
    max_src_len=512, 
    max_tgt_len=128,
    online_augmentation=False
)

# Create enhanced dataloaders
batch_size_aug = 32  # Slightly smaller batch size due to increased data
augmented_train_loader = DataLoader(augmented_train_dataset, batch_size=batch_size_aug, shuffle=True, num_workers=0)
val_loader_enhanced = DataLoader(val_dataset_enhanced, batch_size=batch_size_aug, shuffle=False, num_workers=0)

print(f"✓ Enhanced training batches: {len(augmented_train_loader)}")
print(f"✓ Enhanced validation batches: {len(val_loader_enhanced)}")
print(f"✓ Training samples: {len(augmented_train_dataset)}")
print(f"✓ Validation samples: {len(val_dataset_enhanced)}")

Creating enhanced datasets with augmentation...
✓ Enhanced training batches: 599
✓ Enhanced validation batches: 26
✓ Training samples: 19150
✓ Validation samples: 818


## 26. Initialize New Model for Augmented Training

In [52]:
# Initialize new model for augmented training
print("Initializing new model for augmented training...")

# Enhanced model hyperparameters (slightly improved)
d_model_aug = 512        # Increased embedding dimension
num_layers_aug = 6       # Increased number of layers
num_heads_aug = 8        # Number of attention heads
d_ff_aug = 2048          # Increased feed-forward dimension
dropout_aug = 0.1
vocab_size = len(tokenizer)
pad_idx = tokenizer.pad_token_id

# Initialize enhanced model
model_augmented = Transformer(
    src_vocab_size=vocab_size,
    tgt_vocab_size=vocab_size,
    d_model=d_model_aug,
    num_layers=num_layers_aug,
    num_heads=num_heads_aug,
    d_ff=d_ff_aug,
    dropout=dropout_aug,
    pad_idx=pad_idx
).to(device)

# Option: Load pre-trained weights from previous model
print("Loading pre-trained weights from previous model...")
try:
    # Load the state dict from the previous model
    prev_state_dict = torch.load('best_summarizer_model.pt', map_location=device)
    
    # Try to load compatible parameters
    current_state_dict = model_augmented.state_dict()
    loaded_params = 0
    
    for name, param in prev_state_dict.items():
        if name in current_state_dict and current_state_dict[name].shape == param.shape:
            current_state_dict[name] = param
            loaded_params += 1
    
    model_augmented.load_state_dict(current_state_dict)
    print(f"✓ Loaded {loaded_params} compatible parameters from previous model")
    
except Exception as e:
    print(f"Could not load previous weights: {e}")
    print("✓ Starting with fresh model initialization")

# Count parameters
total_params_aug = sum(p.numel() for p in model_augmented.parameters())
trainable_params_aug = sum(p.numel() for p in model_augmented.parameters() if p.requires_grad)

print(f"\n✓ Enhanced model initialized")
print(f"Total parameters: {total_params_aug:,}")
print(f"Trainable parameters: {trainable_params_aug:,}")
print(f"\nEnhanced model architecture:")
print(f"- Embedding dimension: {d_model_aug}")
print(f"- Encoder/Decoder layers: {num_layers_aug}")
print(f"- Attention heads: {num_heads_aug}")
print(f"- Feed-forward dimension: {d_ff_aug}")
print(f"- Vocabulary size: {vocab_size}")
print(f"- Parameters increase: {(total_params_aug/trainable_params)*100-100:.1f}%")

Initializing new model for augmented training...
Loading pre-trained weights from previous model...
✓ Loaded 1 compatible parameters from previous model

✓ Enhanced model initialized
Total parameters: 93,476,196
Trainable parameters: 93,476,196

Enhanced model architecture:
- Embedding dimension: 512
- Encoder/Decoder layers: 6
- Attention heads: 8
- Feed-forward dimension: 2048
- Vocabulary size: 32100
- Parameters increase: 191.6%
Loading pre-trained weights from previous model...
✓ Loaded 1 compatible parameters from previous model

✓ Enhanced model initialized
Total parameters: 93,476,196
Trainable parameters: 93,476,196

Enhanced model architecture:
- Embedding dimension: 512
- Encoder/Decoder layers: 6
- Attention heads: 8
- Feed-forward dimension: 2048
- Vocabulary size: 32100
- Parameters increase: 191.6%


## 27. Enhanced Training Setup with Learning Rate Scheduling

In [53]:
# Enhanced training setup
print("Setting up enhanced training configuration...")

# Loss function with label smoothing for better generalization
class LabelSmoothingCrossEntropy(nn.Module):
    def __init__(self, smoothing=0.1, ignore_index=-100):
        super(LabelSmoothingCrossEntropy, self).__init__()
        self.smoothing = smoothing
        self.ignore_index = ignore_index
        
    def forward(self, pred, target):
        vocab_size = pred.size(-1)
        pred = pred.log_softmax(dim=-1)
        
        # Create smoothed targets
        with torch.no_grad():
            smooth_target = torch.zeros_like(pred)
            smooth_target.fill_(self.smoothing / (vocab_size - 1))
            smooth_target.scatter_(1, target.unsqueeze(1), 1.0 - self.smoothing)
            
            # Handle ignore_index
            mask = (target == self.ignore_index).unsqueeze(1)
            smooth_target.masked_fill_(mask, 0)
        
        return torch.mean(torch.sum(-smooth_target * pred, dim=1))

# Enhanced loss function
criterion_aug = LabelSmoothingCrossEntropy(smoothing=0.1, ignore_index=pad_idx)

# Enhanced optimizer with different learning rate
learning_rate_aug = 5e-5  # Slightly lower learning rate for fine-tuning
weight_decay = 1e-4

optimizer_aug = torch.optim.AdamW(
    model_augmented.parameters(), 
    lr=learning_rate_aug, 
    betas=(0.9, 0.98), 
    eps=1e-9,
    weight_decay=weight_decay
)

# Learning rate scheduler
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

scheduler = CosineAnnealingWarmRestarts(
    optimizer_aug, 
    T_0=10,  # Restart every 10 epochs
    T_mult=2,  # Double the restart period each time
    eta_min=1e-6  # Minimum learning rate
)

print("✓ Enhanced training setup complete")
print(f"Optimizer: AdamW with weight decay")
print(f"Initial learning rate: {learning_rate_aug}")
print(f"Loss function: Label Smoothing CrossEntropyLoss")
print(f"Scheduler: Cosine Annealing with Warm Restarts")
print(f"Weight decay: {weight_decay}")

Setting up enhanced training configuration...
✓ Enhanced training setup complete
Optimizer: AdamW with weight decay
Initial learning rate: 5e-05
Loss function: Label Smoothing CrossEntropyLoss
Scheduler: Cosine Annealing with Warm Restarts
Weight decay: 0.0001


## 28. Enhanced Training Loop with Augmented Data

In [54]:
def train_epoch_enhanced(model, loader, criterion, optimizer, scheduler, device, epoch):
    """Enhanced training function with better monitoring"""
    model.train()
    total_loss = 0
    total_samples = 0
    
    pbar = tqdm(loader, desc=f"Epoch {epoch} [Train]")
    for batch_idx, batch in enumerate(pbar):
        src = batch['src'].to(device)
        tgt = batch['tgt'].to(device)
        tgt_y = batch['tgt_y'].to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        output = model(src, tgt)
        
        # Reshape for loss calculation
        output = output.reshape(-1, output.shape[-1])
        tgt_y = tgt_y.reshape(-1)
        
        # Calculate loss
        loss = criterion(output, tgt_y)
        
        # Backward pass
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        total_loss += loss.item()
        total_samples += src.size(0)
        
        # Update progress bar
        current_lr = optimizer.param_groups[0]['lr']
        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'avg_loss': f'{total_loss/(batch_idx+1):.4f}',
            'lr': f'{current_lr:.2e}'
        })
    
    return total_loss / len(loader)

def validate_enhanced(model, loader, criterion, device, epoch):
    """Enhanced validation function"""
    model.eval()
    total_loss = 0
    total_samples = 0
    
    with torch.no_grad():
        pbar = tqdm(loader, desc=f"Epoch {epoch} [Val]")
        for batch in pbar:
            src = batch['src'].to(device)
            tgt = batch['tgt'].to(device)
            tgt_y = batch['tgt_y'].to(device)
            
            # Forward pass
            output = model(src, tgt)
            
            # Reshape for loss calculation
            output = output.reshape(-1, output.shape[-1])
            tgt_y = tgt_y.reshape(-1)
            
            # Calculate loss
            loss = criterion(output, tgt_y)
            total_loss += loss.item()
            total_samples += src.size(0)
            
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return total_loss / len(loader)

print("✓ Enhanced training functions defined")

✓ Enhanced training functions defined


In [ ]:
# Enhanced training loop with augmented data
num_epochs_aug = 20  # Training epochs for augmented model
best_val_loss_aug = float('inf')
training_history = {
    'train_loss': [],
    'val_loss': [],
    'learning_rates': []
}

print(f"Starting enhanced training with augmented data for {num_epochs_aug} epochs...")
print(f"Training samples: {len(augmented_train_dataset)}")
print(f"Validation samples: {len(val_dataset_enhanced)}")
print("="*80)

for epoch in range(1, num_epochs_aug + 1):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch}/{num_epochs_aug}")
    print(f"{'='*60}")
    
    # Train
    train_loss = train_epoch_enhanced(
        model_augmented, 
        augmented_train_loader, 
        criterion_aug, 
        optimizer_aug, 
        scheduler,
        device, 
        epoch
    )
    
    # Validate
    val_loss = validate_enhanced(
        model_augmented, 
        val_loader_enhanced, 
        criterion_aug, 
        device, 
        epoch
    )
    
    # Update scheduler
    scheduler.step()
    current_lr = optimizer_aug.param_groups[0]['lr']
    
    # Store history
    training_history['train_loss'].append(train_loss)
    training_history['val_loss'].append(val_loss)
    training_history['learning_rates'].append(current_lr)
    
    print(f"\nEpoch {epoch} Summary:")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val Loss: {val_loss:.4f}")
    print(f"  Learning Rate: {current_lr:.2e}")
    
    # Save best model
    if val_loss < best_val_loss_aug:
        best_val_loss_aug = val_loss
        torch.save(model_augmented.state_dict(), 'best_summarizer_model_augmented.pt')
        print(f"  ✓ Best augmented model saved (val_loss: {val_loss:.4f})")
    
    # Early stopping check
    if epoch > 5:
        recent_val_losses = training_history['val_loss'][-3:]
        if all(recent_val_losses[i] >= recent_val_losses[i-1] for i in range(1, len(recent_val_losses))):
            print(f"  Early stopping triggered at epoch {epoch}")
            break

print("\n" + "="*80)
print("ENHANCED TRAINING COMPLETE!")
print("="*80)
print(f"Best validation loss: {best_val_loss_aug:.4f}")
print(f"Training completed in {epoch} epochs")
print(f"Model saved as: best_summarizer_model_augmented.pt")
print("="*80)

Starting enhanced training with augmented data for 20 epochs...
Training samples: 19150
Validation samples: 818

Epoch 1/20


Epoch 1 [Val]: 100%|██████████| 26/26 [00:03<00:00,  6.80it/s, loss=1.4645]




Epoch 1 Summary:
  Train Loss: 1.5384
  Val Loss: 1.3672
  Learning Rate: 4.88e-05
  ✓ Best augmented model saved (val_loss: 1.3672)

Epoch 2/20
  ✓ Best augmented model saved (val_loss: 1.3672)

Epoch 2/20


Epoch 2 [Val]: 100%|██████████| 26/26 [00:03<00:00,  6.86it/s, loss=1.3577]




Epoch 2 Summary:
  Train Loss: 1.3167
  Val Loss: 1.2616
  Learning Rate: 4.53e-05
  ✓ Best augmented model saved (val_loss: 1.2616)

Epoch 3/20
  ✓ Best augmented model saved (val_loss: 1.2616)

Epoch 3/20


Epoch 3 [Val]: 100%|██████████| 26/26 [00:03<00:00,  6.83it/s, loss=1.3122]




Epoch 3 Summary:
  Train Loss: 1.2383
  Val Loss: 1.2199
  Learning Rate: 3.99e-05
  ✓ Best augmented model saved (val_loss: 1.2199)

Epoch 4/20
  ✓ Best augmented model saved (val_loss: 1.2199)

Epoch 4/20


Epoch 4 [Val]: 100%|██████████| 26/26 [00:03<00:00,  6.83it/s, loss=1.2891]




Epoch 4 Summary:
  Train Loss: 1.1984
  Val Loss: 1.1956
  Learning Rate: 3.31e-05
  ✓ Best augmented model saved (val_loss: 1.1956)

Epoch 5/20
  ✓ Best augmented model saved (val_loss: 1.1956)

Epoch 5/20


Epoch 5 [Train]:  19%|█▊        | 111/599 [00:38<02:50,  2.86it/s, loss=1.2939, avg_loss=1.1655, lr=3.31e-05]

## 29. Test Enhanced Model Performance

In [ ]:
# Load the best augmented model and test performance
print("Loading best augmented model...")
model_augmented.load_state_dict(torch.load('best_summarizer_model_augmented.pt', map_location=device))
model_augmented.eval()
print("✓ Best augmented model loaded")

# Test on validation samples to compare improvement
print("\n" + "="*80)
print("COMPARING ORIGINAL vs AUGMENTED MODEL PERFORMANCE")
print("="*80)

num_test_samples = 5
for i in range(num_test_samples):
    sample = dataset['validation'][i]
    dialogue = sample['dialogue']
    ground_truth = sample['summary']
    
    print(f"\n{'='*60}")
    print(f"Test Sample {i+1}")
    print(f"{'='*60}")
    print(f"\n📝 Dialogue:\n{dialogue[:300]}...")
    print(f"\n✅ Ground Truth:\n{ground_truth}")
    
    # Generate with original model
    original_summary = generate_summary(model, dialogue, tokenizer, device)
    print(f"\n🤖 Original Model:\n{original_summary}")
    
    # Generate with augmented model  
    augmented_summary = generate_summary(model_augmented, dialogue, tokenizer, device)
    print(f"\n🚀 Augmented Model:\n{augmented_summary}")
    
    # Quick ROUGE comparison
    rouge_original = evaluator.calculate_rouge_scores(ground_truth, original_summary)
    rouge_augmented = evaluator.calculate_rouge_scores(ground_truth, augmented_summary)
    
    print(f"\n📊 ROUGE-L F1 Scores:")
    print(f"   Original:  {rouge_original['rougeL_fmeasure']:.4f}")
    print(f"   Augmented: {rouge_augmented['rougeL_fmeasure']:.4f}")
    print(f"   Improvement: {rouge_augmented['rougeL_fmeasure'] - rouge_original['rougeL_fmeasure']:+.4f}")
    print(f"\n{'-'*60}")

print(f"\n{'='*80}")
print("PERFORMANCE COMPARISON COMPLETE")
print(f"{'='*80}")

## 30. Training History Visualization

In [ ]:
import matplotlib.pyplot as plt

# Plot training history
print("Visualizing training progress...")

plt.figure(figsize=(15, 5))

# Plot 1: Training and Validation Loss
plt.subplot(1, 3, 1)
epochs = range(1, len(training_history['train_loss']) + 1)
plt.plot(epochs, training_history['train_loss'], 'b-', label='Training Loss')
plt.plot(epochs, training_history['val_loss'], 'r-', label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plot 2: Learning Rate Schedule
plt.subplot(1, 3, 2)
plt.plot(epochs, training_history['learning_rates'], 'g-')
plt.title('Learning Rate Schedule')
plt.xlabel('Epoch')
plt.ylabel('Learning Rate')
plt.yscale('log')
plt.grid(True)

# Plot 3: Validation Loss Improvement
plt.subplot(1, 3, 3)
best_val_losses = []
best_so_far = float('inf')
for val_loss in training_history['val_loss']:
    if val_loss < best_so_far:
        best_so_far = val_loss
    best_val_losses.append(best_so_far)

plt.plot(epochs, training_history['val_loss'], 'r-', alpha=0.7, label='Validation Loss')
plt.plot(epochs, best_val_losses, 'g-', linewidth=2, label='Best Validation Loss')
plt.title('Validation Loss Progress')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# Print training summary
print("\n" + "="*60)
print("TRAINING SUMMARY")
print("="*60)
print(f"Final Training Loss: {training_history['train_loss'][-1]:.4f}")
print(f"Final Validation Loss: {training_history['val_loss'][-1]:.4f}")
print(f"Best Validation Loss: {min(training_history['val_loss']):.4f}")
print(f"Total Epochs: {len(training_history['train_loss'])}")
print(f"Final Learning Rate: {training_history['learning_rates'][-1]:.2e}")

# Calculate improvement over original model
original_best_loss = best_val_loss  # From original training
augmented_best_loss = best_val_loss_aug
improvement = ((original_best_loss - augmented_best_loss) / original_best_loss) * 100

print(f"\n📈 MODEL IMPROVEMENT:")
print(f"Original Best Loss: {original_best_loss:.4f}")
print(f"Augmented Best Loss: {augmented_best_loss:.4f}")
print(f"Improvement: {improvement:.2f}%")
print("="*60)